In [1]:
from google.colab import drive
drive.mount('/content/drive')
import shutil

Mounted at /content/drive


In [2]:
import os
import shutil
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Download
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    shutil.move("/content/UCI HAR Dataset", DATA_PATH)

    print("Download completed")

# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    return X, y


def load_subjects(split):
    """샘플별 피험자 ID 로드 (X, y와 동일한 순서로 대응됨)"""
    path = f"{DATA_PATH}/{split}/subject_{split}.txt"
    return np.loadtxt(path).astype(int)


# Official train/test split
X_train, y_train = load_data("train")
X_test, y_test = load_data("test")

# 피험자 ID (subject 기반 val 분리에 사용)
subject_train = load_subjects("train")


print("\nDataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("subject_train:", subject_train.shape)

Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)
subject_train: (7352,)


In [3]:
# ============================================================
# 5. Normalization
# ============================================================

mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

In [4]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)

In [10]:
from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# 1. Group K-fold 설정 (피험자 단위로 겹치지 않게 분할)
# ------------------------------------------------------------

N_FOLDS = 5

gkf_full = GroupKFold(n_splits=N_FOLDS)

class_weight_full = {
    0: 1.0,   # WALKING
    1: 1.0,   # WALKING_UPSTAIRS
    2: 1.0,   # WALKING_DOWNSTAIRS
    3: 2.0,   # SITTING
    4: 1.0,   # STANDING
    5: 1.0,   # LAYING
}

class_names = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
               "SITTING", "STANDING", "LAYING"]

fold_models_full = []
fold_val_accuracies_full = []
test_probs_full = []   # 각 fold 모델의 X_test 예측 확률을 저장 (나중에 평균)


# ------------------------------------------------------------
# 2. Fold별 학습
# ------------------------------------------------------------

for fold_idx, (train_idx, val_idx) in enumerate(
    gkf_full.split(X_train, y_train, groups=subject_train)
):

    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_fold = X_train[train_idx]
    y_train_fold = y_train_onehot[train_idx]

    X_val_fold = X_train[val_idx]
    y_val_fold = y_train_onehot[val_idx]

    print(f"Train subjects: {len(np.unique(subject_train[train_idx]))}, "
          f"Val subjects: {len(np.unique(subject_train[val_idx]))}")
    print(f"X_train_fold: {X_train_fold.shape}, X_val_fold: {X_val_fold.shape}")

    # 모델 정의 (기존 구조 그대로 재사용)
    inputs_full = Input(shape=(128, 9))
    conv1_full = Conv1D(filters=32, kernel_size=5, activation="relu")(inputs_full)
    pool1_full = MaxPooling1D(pool_size=2)(conv1_full)
    conv2_full = Conv1D(filters=64, kernel_size=3, activation="relu")(pool1_full)
    pool2_full = MaxPooling1D(pool_size=2)(conv2_full)
    flat_full = Flatten()(pool2_full)
    drop1_full = Dropout(0.5)(flat_full)
    dense_full = Dense(128, activation="relu", kernel_regularizer=l2(0.009))(drop1_full)
    drop2_full = Dropout(0.3)(dense_full)
    outputs_full = Dense(6, activation="softmax")(drop2_full)

    model_fold = Model(inputs=inputs_full, outputs=outputs_full)

    model_fold.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    history_fold = model_fold.fit(
        X_train_fold,
        y_train_fold,
        epochs=20,
        batch_size=64,
        validation_data=(X_val_fold, y_val_fold),
        shuffle=True,
        class_weight=class_weight_full,
        verbose=1
    )

    fold_models_full.append(model_fold)
    fold_val_accuracies_full.append(history_fold.history["val_accuracy"][-1])

    # 이 fold 모델로 X_test 예측 확률 저장 (나중에 앙상블 평균용)
    test_probs_full.append(model_fold.predict(X_test, verbose=0))


print("\n" + "=" * 60)
print("FOLD별 최종 val_accuracy:", [f"{a:.4f}" for a in fold_val_accuracies_full])
print("평균 val_accuracy:", f"{np.mean(fold_val_accuracies_full):.4f}")


# ------------------------------------------------------------
# 3. 앙상블 예측 (5개 fold 모델의 확률 평균)
# ------------------------------------------------------------

y_prob_full = np.mean(test_probs_full, axis=0)
y_pred_full = np.argmax(y_prob_full, axis=1)


# ------------------------------------------------------------
# 4. 평가
# ------------------------------------------------------------

accuracy_full = accuracy_score(y_test, y_pred_full)
precision_full = precision_score(y_test, y_pred_full, average="macro")
recall_full = recall_score(y_test, y_pred_full, average="macro")
f1_full = f1_score(y_test, y_pred_full, average="macro")

print("\n==============================")
print("6-CLASS TEST RESULTS (K-fold Ensemble)")
print("==============================")
print(f"Accuracy  : {accuracy_full:.4f}")
print(f"Precision : {precision_full:.4f}  (macro avg)")
print(f"Recall    : {recall_full:.4f}  (macro avg)")
print(f"F1-score  : {f1_full:.4f}  (macro avg)")

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")
print(classification_report(
    y_test,
    y_pred_full,
    target_names=class_names,
    digits=4
))

print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")
print(confusion_matrix(y_test, y_pred_full))


FOLD 1/5
Train subjects: 17, Val subjects: 4
X_train_fold: (5932, 128, 9), X_val_fold: (1420, 128, 9)
Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.7653 - loss: 1.9693 - val_accuracy: 0.8683 - val_loss: 0.9813
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9332 - loss: 0.6944 - val_accuracy: 0.8775 - val_loss: 0.5896
Epoch 3/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9415 - loss: 0.3902 - val_accuracy: 0.8944 - val_loss: 0.4309
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9474 - loss: 0.2967 - val_accuracy: 0.8796 - val_loss: 0.3807
Epoch 5/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9496 - loss: 0.2521 - val_accuracy: 0.8901 - val_loss: 0.3653
Epoch 6/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9523 - loss: 0.2375 - val_accuracy: 0.8817 - val_loss: 0.4420
Epoch 7/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9509 - loss: 0.2198 - val_accuracy: 0.8697 - val_loss: 0.6984
Epoch 8/20
93/

In [5]:
# ============================================================
# SITTING vs STANDING 이진분류 모델 (Dropout + Subject 기반 Val 분리)
# ============================================================

# ------------------------------------------------------------
# 1. 데이터 필터링 (SITTING=3, STANDING=4만 추출, 0/1 재매핑)
# ------------------------------------------------------------

train_mask = np.isin(y_train, [3, 4])
test_mask = np.isin(y_test, [3, 4])

X_train_bin = X_train[train_mask]
y_train_bin = y_train[train_mask]
subject_train_bin = subject_train[train_mask]   # X_train_bin과 1:1 대응하는 피험자 ID

X_test_bin = X_test[test_mask]
y_test_bin = y_test[test_mask]

# 라벨 재매핑: SITTING(3) -> 0, STANDING(4) -> 1
y_train_bin = (y_train_bin == 4).astype(int)
y_test_bin = (y_test_bin == 4).astype(int)

print("SITTING/STANDING 데이터 shape")
print("X_train_bin:", X_train_bin.shape)
print("y_train_bin:", y_train_bin.shape, " (0=SITTING, 1=STANDING)")
print("X_test_bin :", X_test_bin.shape)
print("y_test_bin :", y_test_bin.shape)


# ------------------------------------------------------------
# 2. Subject 기준 Train/Val 분리
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

unique_subjects = np.unique(subject_train_bin)
n_val_subjects = max(1, int(len(unique_subjects) * 0.2))
val_subjects = rng.choice(unique_subjects, size=n_val_subjects, replace=False)

val_mask = np.isin(subject_train_bin, val_subjects)

X_val_bin = X_train_bin[val_mask]
y_val_bin = y_train_bin[val_mask]

X_train_bin_final = X_train_bin[~val_mask]
y_train_bin_final = y_train_bin[~val_mask]

print("\nVal 피험자 목록:", sorted(val_subjects.tolist()))
print("X_train_bin_final:", X_train_bin_final.shape)
print("X_val_bin        :", X_val_bin.shape)


SITTING/STANDING 데이터 shape
X_train_bin: (2660, 128, 9)
y_train_bin: (2660,)  (0=SITTING, 1=STANDING)
X_test_bin : (1023, 128, 9)
y_test_bin : (1023,)

Val 피험자 목록: [3, 16, 22, 23]
X_train_bin_final: (2139, 128, 9)
X_val_bin        : (521, 128, 9)


In [ ]:
# @title
import matplotlib.pyplot as plt

signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING"
]

target_labels = [3,4]   # 여기 리스트만 바꾸면 원하는 라벨만 그려짐

colors = plt.cm.tab10(np.linspace(0, 1, 6))

n_samples_per_class = 20
first_sample = 0

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for ch_idx, ch_name in enumerate(signals):
    ax = axes[ch_idx]

    for label in target_labels:
        indices = np.where(y_train == label)[0]
        selected = indices[first_sample:first_sample + n_samples_per_class]

        for i, idx in enumerate(selected):
            ax.plot(
                X_train[idx, :, ch_idx],
                color=colors[label],
                alpha=0.6,
                label=class_names[label] if i == 0 else None
            )

    ax.set_title(ch_name)
    ax.set_xlabel("Timestep (128 = 2.56s)")
    ax.set_ylabel("normalized value")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.suptitle("Signal pattern by feature (selected activities)", y=1.02)
plt.show()

In [ ]:
# @title
import matplotlib.pyplot as plt
import numpy as np

signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING"
]

target_labels = [3, 4]   # 여기 리스트만 바꾸면 원하는 라벨만 비교 가능

# ------------------------------------------------------------
# 1. 각 샘플(윈도우)을 128 타임스텝 평균 하나의 값으로 축약
# ------------------------------------------------------------
# X_train shape: (N, 128, 9) -> 타임스텝 축(axis=1) 평균 -> (N, 9)
X_train_mean = X_train.mean(axis=1)

# ------------------------------------------------------------
# 2. Boxplot: 채널별 subplot, 그 안에 라벨별 분포 비교
# ------------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for ch_idx, ch_name in enumerate(signals):
    ax = axes[ch_idx]

    data_per_label = []
    for label in target_labels:
        indices = np.where(y_train == label)[0]
        values = X_train_mean[indices, ch_idx]
        data_per_label.append(values)

    ax.boxplot(
        data_per_label,
        labels=[class_names[l] for l in target_labels],
        showmeans=True
    )

    ax.set_title(ch_name)
    ax.set_ylabel("mean value per window (normalized)")

plt.tight_layout()
plt.suptitle("Feature distribution by activity (window mean)", y=1.02)
plt.show()

# ------------------------------------------------------------
# 3. 숫자 표로 평균/표준편차 확인
# ------------------------------------------------------------
print(f"{'feature':15s}", end="")
for label in target_labels:
    print(f"| {class_names[label]:>10s}(mean) {class_names[label]:>10s}(std) ", end="")
print()
print("-" * (15 + 30 * len(target_labels)))

for ch_idx, ch_name in enumerate(signals):
    print(f"{ch_name:15s}", end="")
    for label in target_labels:
        indices = np.where(y_train == label)[0]
        values = X_train_mean[indices, ch_idx]
        print(f"| {values.mean():18.4f} {values.std():18.4f} ", end="")
    print()